In [10]:
from pathlib import Path

from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

from bst_bev.maps.bst_map_index import BSTMapIndex
from bst_bev.query.fixed_zoom import FixedZoomRegionCover
import time


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


DATAROOT = (
    PROJECT_ROOT
    / "data"
    / "nuscenes"
)

RADIUS_M = 30.0

In [11]:
nusc = NuScenes(
    version="v1.0-mini",
    dataroot=str(DATAROOT),
    verbose=False,
)

scene = nusc.scene[0]

log = nusc.get(
    "log",
    scene["log_token"],
)

location = log["location"]

nusc_map = NuScenesMap(
    dataroot=str(DATAROOT),
    map_name=location,
)

sample = nusc.get(
    "sample",
    scene["first_sample_token"],
)

lidar = nusc.get(
    "sample_data",
    sample["data"]["LIDAR_TOP"],
)

ego_pose = nusc.get(
    "ego_pose",
    lidar["ego_pose_token"],
)

ego_x, ego_y, _ = (
    ego_pose["translation"]
)

print("Scene:", scene["name"])
print("Location:", location)
print("Ego:", ego_x, ego_y)

Scene: scene-0061
Location: singapore-onenorth
Ego: 411.3039349319818 1180.8903791765097


In [12]:
MAP_LAYERS = [
    "lane_divider",
    "road_divider",
    "ped_crossing",
]

patch = (
    ego_x - RADIUS_M,
    ego_y - RADIUS_M,
    ego_x + RADIUS_M,
    ego_y + RADIUS_M,
)

records = nusc_map.get_records_in_patch(
    patch,
    MAP_LAYERS,
    mode="intersect",
)

gt_tokens = {
    (layer, token)
    for layer, tokens in records.items()
    for token in tokens
}

In [13]:
def compare_queries(zoom):
    index = BSTMapIndex(
        nusc_map=nusc_map,
        location=location,
        zoom=zoom,
        index_spacing_m=1.0,
        adaptive_index_spacing=False,
    )

    index.build()

    # --------------------------------------------
    # Existing sampling-based method
    # --------------------------------------------

    start = time.perf_counter()

    sampling_result = index.query_patch(
        center_x=ego_x,
        center_y=ego_y,
        radius=RADIUS_M,
        step_m=None,
    )

    sampling_time_ms = (
        time.perf_counter() - start
    ) * 1000

    sampling_tokens = (
        sampling_result.tokens
    )

    # --------------------------------------------
    # New direct method
    # --------------------------------------------

    region_cover = (
        FixedZoomRegionCover(
            location=location,
            zoom=zoom,
        )
    )

    start = time.perf_counter()

    direct_cover = region_cover.cover(
        center_x=ego_x,
        center_y=ego_y,
        radius=RADIUS_M,
    )

    direct_tokens = (
        index.query_cell_tokens(
            direct_cover.keys
        )
    )

    direct_time_ms = (
        time.perf_counter() - start
    ) * 1000

    direct_correct = (
        direct_tokens
        & gt_tokens
    )

    direct_precision = (
        len(direct_correct)
        / len(direct_tokens)
        if direct_tokens
        else 0.0
    )

    direct_recall = (
        len(direct_correct)
        / len(gt_tokens)
        if gt_tokens
        else 0.0
    )

    # --------------------------------------------
    # Compare
    # --------------------------------------------

    token_intersection = (
        sampling_tokens
        & direct_tokens
    )

    token_union = (
        sampling_tokens
        | direct_tokens
    )

    jaccard = (
        len(token_intersection)
        / len(token_union)
        if token_union
        else 1.0
    )

    return {
        "zoom": zoom,

        "sampling_cells": (
            sampling_result.cell_count
        ),

        "direct_cells": (
            direct_cover.cell_count
        ),

        "sampling_candidates": (
            len(sampling_tokens)
        ),

        "direct_candidates": (
            len(direct_tokens)
        ),

        "same_candidates": (
            sampling_tokens
            == direct_tokens
        ),

        "candidate_jaccard": jaccard,

        "sampling_only": (
            sampling_tokens
            - direct_tokens
        ),

        "direct_only": (
            direct_tokens
            - sampling_tokens
        ),
        "direct_precision": (
            direct_precision
        ),

        "direct_recall": (
            direct_recall
        ),

        "sampling_time_ms": (
            sampling_time_ms
        ),

        "direct_time_ms": (
            direct_time_ms
        ),

        "speedup": (
            sampling_time_ms
            / direct_time_ms
            if direct_time_ms > 0
            else float("inf")
        ),
    }

In [14]:
result_23 = compare_queries(23)
result_23

{'zoom': 23,
 'sampling_cells': 196,
 'direct_cells': 196,
 'sampling_candidates': 4,
 'direct_candidates': 4,
 'same_candidates': True,
 'candidate_jaccard': 1.0,
 'sampling_only': set(),
 'direct_only': set(),
 'direct_precision': 1.0,
 'direct_recall': 1.0,
 'sampling_time_ms': 11.308417189866304,
 'direct_time_ms': 1.1123749427497387,
 'speedup': 10.166012155858551}

In [15]:
result_24 = compare_queries(24)
result_24

{'zoom': 24,
 'sampling_cells': 702,
 'direct_cells': 702,
 'sampling_candidates': 4,
 'direct_candidates': 4,
 'same_candidates': True,
 'candidate_jaccard': 1.0,
 'sampling_only': set(),
 'direct_only': set(),
 'direct_precision': 1.0,
 'direct_recall': 1.0,
 'sampling_time_ms': 11.538834078237414,
 'direct_time_ms': 3.918624948710203,
 'speedup': 2.944613028617441}